# Step 1: USDA Data Preprocessing

This notebook handles the preprocessing of USDA food data:
1. Load food.csv and branded_food.csv
2. Clean ingredient lists and handle missing values
3. Merge datasets on fdc_id
4. Save cleaned_usda_data.csv

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import re
import os

## 1. Load USDA Datasets

In [3]:
# Define file paths (relative to notebooks folder)
USDA_DATA_PATH = '../data/raw/usda/usda_fdc/'
OUTPUT_PATH = '../data/pre-processed/usda/'

# Ensure output directory exists
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Loading food.csv...")
food_df = pd.read_csv(os.path.join(USDA_DATA_PATH, 'food.csv'), low_memory=False)
print(f"Food dataset shape: {food_df.shape}")

print("\nLoading branded_food.csv...")
branded_df = pd.read_csv(os.path.join(USDA_DATA_PATH, 'branded_food.csv'), low_memory=False)
print(f"Branded food dataset shape: {branded_df.shape}")

Loading food.csv...
Food dataset shape: (2064912, 5)

Loading branded_food.csv...
Branded food dataset shape: (1977398, 21)


In [4]:
# Inspect food dataset
print("Food Dataset Columns:")
print(food_df.columns.tolist())
print("\nFirst 5 rows:")
food_df.head()

Food Dataset Columns:
['fdc_id', 'data_type', 'description', 'food_category_id', 'publication_date']

First 5 rows:


,fdc_id,data_type,description,food_category_id,publication_date
0,1105904,branded_food,WESSON Vegetable Oil 1 GAL,Oils Edible,2020-11-13
1,1105905,branded_food,SWANSON BROTH BEEF,Herbs/Spices/Extracts,2020-11-13
2,1105906,branded_food,CAMPBELL'S SLOW KETTLE SOUP CLAM CHOWDER,Prepared Soups,2020-11-13
3,1105907,branded_food,CAMPBELL'S SLOW KETTLE SOUP CHEESE BROCCOLI,Prepared Soups,2020-11-13
4,1105898,experimental_food,Discrepancy between the Atwater factor predict...,NaN,2020-10-30


In [5]:
# Inspect branded food dataset
print("Branded Food Dataset Columns:")
print(branded_df.columns.tolist())
print("\nFirst 5 rows:")
branded_df.head()

Branded Food Dataset Columns:
['fdc_id', 'brand_owner', 'brand_name', 'subbrand_name', 'gtin_upc', 'ingredients', 'not_a_significant_source_of', 'serving_size', 'serving_size_unit', 'household_serving_fulltext', 'branded_food_category', 'data_source', 'package_weight', 'modified_date', 'available_date', 'market_country', 'discontinued_date', 'preparation_state_code', 'trade_channel', 'short_description', 'material_code']

First 5 rows:


,fdc_id,brand_owner,brand_name,subbrand_name,gtin_upc,ingredients,not_a_significant_source_of,serving_size,serving_size_unit,household_serving_fulltext,...,data_source,package_weight,modified_date,available_date,market_country,discontinued_date,preparation_state_code,trade_channel,short_description,material_code
0,1105904,Richardson Oilseed Products (US) Limited,NaN,NaN,00027000612323,Vegetable Oil,NaN,15.0,ml,1 Tbsp (15 ml),...,GDSN,NaN,2020-10-02,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN
1,1105905,CAMPBELL SOUP COMPANY,NaN,NaN,00051000198808,"INGREDIENTS: BEEF STOCK, CONTAINS LESS THAN 2%...",NaN,240.0,ml,Amount per serving,...,GDSN,NaN,2020-09-12,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN
2,1105906,CAMPBELL SOUP COMPANY,NaN,NaN,00051000213273,"INGREDIENTS: CLAM STOCK, POTATOES, CLAMS, CREA...",NaN,440.0,g,PER CONTAINER,...,GDSN,NaN,2020-09-01,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN
3,1105907,CAMPBELL SOUP COMPANY,NaN,NaN,00051000213303,"INGREDIENTS: WATER, CREAM, BROCCOLI, CELERY, V...",NaN,440.0,g,PER CONTAINER,...,GDSN,NaN,2020-09-01,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN
4,1105908,CAMPBELL SOUP COMPANY,NaN,NaN,00051000224637,"INGREDIENTS: CHICKEN STOCK, CONTAINS LESS THAN...",NaN,240.0,ml,Amount per Serving,...,GDSN,NaN,2020-10-03,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN


## 2. Data Exploration and Quality Check

In [6]:
# Check missing values in food dataset
print("Missing values in food dataset:")
print(food_df.isnull().sum())
print(f"\nTotal rows: {len(food_df)}")

Missing values in food dataset:
fdc_id                  0
data_type               0
description             9
food_category_id    14475
publication_date        0
dtype: int64

Total rows: 2064912


In [7]:
# Check missing values in branded food dataset
print("Missing values in branded food dataset:")
print(branded_df.isnull().sum())
print(f"\nTotal rows: {len(branded_df)}")

Missing values in branded food dataset:
fdc_id                               0
brand_owner                      17232
brand_name                      544896
subbrand_name                  1871607
gtin_upc                             0
ingredients                       5373
not_a_significant_source_of    1897783
serving_size                     10753
serving_size_unit                18998
household_serving_fulltext       41309
branded_food_category            10630
data_source                          0
package_weight                 1133565
modified_date                       20
available_date                       0
market_country                       0
discontinued_date              1974152
preparation_state_code         1895488
trade_channel                  1943460
short_description              1895391
material_code                  1976075
dtype: int64

Total rows: 1977398


In [8]:
# Check data types
print("Food dataset data types:")
print(food_df.dtypes)
print("\nBranded food dataset data types:")
print(branded_df.dtypes)

Food dataset data types:
fdc_id               int64
data_type           object
description         object
food_category_id    object
publication_date    object
dtype: object

Branded food dataset data types:
fdc_id                           int64
brand_owner                     object
brand_name                      object
subbrand_name                   object
gtin_upc                        object
ingredients                     object
not_a_significant_source_of     object
serving_size                   float64
serving_size_unit               object
household_serving_fulltext      object
branded_food_category           object
data_source                     object
package_weight                  object
modified_date                   object
available_date                  object
market_country                  object
discontinued_date               object
preparation_state_code          object
trade_channel                   object
short_description               object
material_cod

## 3. Clean Ingredient Lists

In [9]:
def clean_ingredient_text(text):
    """
    Clean ingredient text by:
    - Converting to lowercase
    - Removing extra whitespace
    - Removing special characters (keeping commas for separation)
    - Stripping leading/trailing whitespace
    """
    if pd.isna(text) or text == '':
        return ''
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove extra whitespace characters
    text = re.sub(r'\s+', ' ', text)
    
    # Remove special characters but keep essential punctuation
    text = re.sub(r'[^a-z0-9\s,().%-]', '', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text

# Apply cleaning to ingredients column if it exists
if 'ingredients' in branded_df.columns:
    print("Cleaning ingredient lists in branded food dataset...")
    branded_df['ingredients_cleaned'] = branded_df['ingredients'].apply(clean_ingredient_text)
    print("Sample cleaned ingredients:")
    print(branded_df[['ingredients', 'ingredients_cleaned']].head())

Cleaning ingredient lists in branded food dataset...
Sample cleaned ingredients:
                                         ingredients  \
0                                      Vegetable Oil   
1  INGREDIENTS: BEEF STOCK, CONTAINS LESS THAN 2%...   
2  INGREDIENTS: CLAM STOCK, POTATOES, CLAMS, CREA...   
3  INGREDIENTS: WATER, CREAM, BROCCOLI, CELERY, V...   
4  INGREDIENTS: CHICKEN STOCK, CONTAINS LESS THAN...   

                                 ingredients_cleaned  
0                                      vegetable oil  
1  ingredients beef stock, contains less than 2% ...  
2  ingredients clam stock, potatoes, clams, cream...  
3  ingredients water, cream, broccoli, celery, ve...  
4  ingredients chicken stock, contains less than ...  


## 4. Handle Missing Values

In [10]:
# Handle missing values in food dataset
print("Handling missing values in food dataset...")

# Fill description with 'Unknown' if missing
if 'description' in food_df.columns:
    food_df['description'] = food_df['description'].fillna('Unknown')

# Fill food_category_id with -1 if missing (to indicate unknown category)
if 'food_category_id' in food_df.columns:
    food_df['food_category_id'] = food_df['food_category_id'].fillna(-1)

print("Missing values after handling (food):")
print(food_df.isnull().sum())

Handling missing values in food dataset...
Missing values after handling (food):
fdc_id              0
data_type           0
description         0
food_category_id    0
publication_date    0
dtype: int64


In [11]:
# Handle missing values in branded food dataset
print("Handling missing values in branded food dataset...")

# Define columns and their fill values
fill_values = {
    'brand_owner': 'Unknown',
    'brand_name': 'Unknown',
    'ingredients': '',
    'ingredients_cleaned': '',
    'serving_size': 0,
    'serving_size_unit': 'Unknown',
    'branded_food_category': 'Unknown',
    'gtin_upc': 'Unknown'
}

for col, fill_val in fill_values.items():
    if col in branded_df.columns:
        branded_df[col] = branded_df[col].fillna(fill_val)

print("Missing values after handling (branded food - top columns):")
print(branded_df.isnull().sum().head(20))

Handling missing values in branded food dataset...
Missing values after handling (branded food - top columns):
fdc_id                               0
brand_owner                          0
brand_name                           0
subbrand_name                  1871607
gtin_upc                             0
ingredients                          0
not_a_significant_source_of    1897783
serving_size                         0
serving_size_unit                    0
household_serving_fulltext       41309
branded_food_category                0
data_source                          0
package_weight                 1133565
modified_date                       20
available_date                       0
market_country                       0
discontinued_date              1974152
preparation_state_code         1895488
trade_channel                  1943460
short_description              1895391
dtype: int64


## 5. Merge Datasets on fdc_id

In [12]:
# Check common key
print(f"Unique fdc_id in food: {food_df['fdc_id'].nunique()}")
print(f"Unique fdc_id in branded_food: {branded_df['fdc_id'].nunique()}")

# Find common fdc_ids
common_ids = set(food_df['fdc_id']).intersection(set(branded_df['fdc_id']))
print(f"\nCommon fdc_ids: {len(common_ids)}")

Unique fdc_id in food: 2064912
Unique fdc_id in branded_food: 1977398

Common fdc_ids: 1977398


In [13]:
# Merge datasets on fdc_id
print("Merging food and branded_food datasets on fdc_id...")

# Left join to keep all food items, adding branded food info where available
merged_df = food_df.merge(
    branded_df,
    on='fdc_id',
    how='left',
    suffixes=('', '_branded')
)

print(f"\nMerged dataset shape: {merged_df.shape}")
print(f"Columns in merged dataset: {merged_df.columns.tolist()}")

Merging food and branded_food datasets on fdc_id...

Merged dataset shape: (2064912, 26)
Columns in merged dataset: ['fdc_id', 'data_type', 'description', 'food_category_id', 'publication_date', 'brand_owner', 'brand_name', 'subbrand_name', 'gtin_upc', 'ingredients', 'not_a_significant_source_of', 'serving_size', 'serving_size_unit', 'household_serving_fulltext', 'branded_food_category', 'data_source', 'package_weight', 'modified_date', 'available_date', 'market_country', 'discontinued_date', 'preparation_state_code', 'trade_channel', 'short_description', 'material_code', 'ingredients_cleaned']


In [14]:
# View sample of merged data
print("Sample of merged data:")
merged_df.head()

Sample of merged data:


,fdc_id,data_type,description,food_category_id,publication_date,brand_owner,brand_name,subbrand_name,gtin_upc,ingredients,...,package_weight,modified_date,available_date,market_country,discontinued_date,preparation_state_code,trade_channel,short_description,material_code,ingredients_cleaned
0,1105904,branded_food,WESSON Vegetable Oil 1 GAL,Oils Edible,2020-11-13,Richardson Oilseed Products (US) Limited,Unknown,NaN,00027000612323,Vegetable Oil,...,NaN,2020-10-02,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN,vegetable oil
1,1105905,branded_food,SWANSON BROTH BEEF,Herbs/Spices/Extracts,2020-11-13,CAMPBELL SOUP COMPANY,Unknown,NaN,00051000198808,"INGREDIENTS: BEEF STOCK, CONTAINS LESS THAN 2%...",...,NaN,2020-09-12,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN,"ingredients beef stock, contains less than 2% ..."
2,1105906,branded_food,CAMPBELL'S SLOW KETTLE SOUP CLAM CHOWDER,Prepared Soups,2020-11-13,CAMPBELL SOUP COMPANY,Unknown,NaN,00051000213273,"INGREDIENTS: CLAM STOCK, POTATOES, CLAMS, CREA...",...,NaN,2020-09-01,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN,"ingredients clam stock, potatoes, clams, cream..."
3,1105907,branded_food,CAMPBELL'S SLOW KETTLE SOUP CHEESE BROCCOLI,Prepared Soups,2020-11-13,CAMPBELL SOUP COMPANY,Unknown,NaN,00051000213303,"INGREDIENTS: WATER, CREAM, BROCCOLI, CELERY, V...",...,NaN,2020-09-01,2020-11-13,United States,NaN,NaN,NaN,NaN,NaN,"ingredients water, cream, broccoli, celery, ve..."
4,1105898,experimental_food,Discrepancy between the Atwater factor predict...,-1,2020-10-30,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Final Cleaning and Data Validation

In [15]:
# Clean text columns - strip whitespace and normalize
text_columns = merged_df.select_dtypes(include=['object']).columns

for col in text_columns:
    merged_df[col] = merged_df[col].str.strip() if merged_df[col].dtype == 'object' else merged_df[col]
    merged_df[col] = merged_df[col].str.replace(r'\s+', ' ', regex=True) if merged_df[col].dtype == 'object' else merged_df[col]

print("Text columns cleaned and normalized.")

Text columns cleaned and normalized.


In [16]:
# Final data quality check
print("Final Dataset Summary:")
print(f"Total rows: {len(merged_df)}")
print(f"Total columns: {len(merged_df.columns)}")
print("\nData types:")
print(merged_df.dtypes)
print("\nMissing values summary:")
print(merged_df.isnull().sum())

Final Dataset Summary:
Total rows: 2064912
Total columns: 26

Data types:
fdc_id                           int64
data_type                       object
description                     object
food_category_id                object
publication_date                object
brand_owner                     object
brand_name                      object
subbrand_name                   object
gtin_upc                        object
ingredients                     object
not_a_significant_source_of     object
serving_size                   float64
serving_size_unit               object
household_serving_fulltext      object
branded_food_category           object
data_source                     object
package_weight                  object
modified_date                   object
available_date                  object
market_country                  object
discontinued_date               object
preparation_state_code          object
trade_channel                   object
short_description            

## 7. Save Cleaned Dataset

In [17]:
# Save the cleaned and merged dataset
output_file = os.path.join(OUTPUT_PATH, 'cleaned_usda_data.csv')

print(f"Saving cleaned data to {output_file}...")
merged_df.to_csv(output_file, index=False)

# Verify file was saved
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file) / (1024 * 1024)  # Convert to MB
    print(f"\nFile saved successfully!")
    print(f"File size: {file_size:.2f} MB")
else:
    print("Error: File was not saved!")

Saving cleaned data to ../data/pre-processed/usda/cleaned_usda_data.csv...

File saved successfully!
File size: 1542.73 MB


In [18]:
# Summary statistics
print("\n" + "="*50)
print("USDA Data Preprocessing Complete!")
print("="*50)
print(f"\nInput files:")
print(f"  - food.csv: {len(food_df)} records")
print(f"  - branded_food.csv: {len(branded_df)} records")
print(f"\nOutput file:")
print(f"  - cleaned_usda_data.csv: {len(merged_df)} records")
print(f"\nColumns in output: {len(merged_df.columns)}")


USDA Data Preprocessing Complete!

Input files:
  - food.csv: 2064912 records
  - branded_food.csv: 1977398 records

Output file:
  - cleaned_usda_data.csv: 2064912 records

Columns in output: 26
